# Mission 03: Common Tools Integration - 해답 노트북

이 노트북은 세 번째 미션의 완성된 솔루션 코드와 설명입니다.

In [ ]:
# 1. 필요한 라이브러리 및 환경 로드
import sys
import os
from dotenv import load_dotenv

while not os.path.exists("app") and os.getcwd() != "/":
    os.chdir("..")
# 루트 폴더 기준의 경로 등록
sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("."))  # sys.path.append("app") 대신 "." 등록이 파이썬 패키지 경로 탐색에 안전합니다.
load_dotenv(override=True)

from app.utils.llm import get_llm
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 범용 도구 임포트
from app.tools import (
    tools_chatbot, file_read, bash_command, glob_search, web_search
)

### [미션 1] 도구 명세 확인하기

도구들의 구조와 정의 형식을 확인하여 어떻게 LLM에게 바인딩되는지 확인합니다.

In [ ]:
# file_read 도구의 명세 출력
print("Tool Name:", file_read.name)
print("Description:", file_read.description)
print("Arguments Schema:", file_read.args)

print("\n" + "-"*50 + "\n")

# bash_command 도구의 명세 출력
print("Tool Name:", bash_command.name)
print("Description:", bash_command.description)
print("Arguments Schema:", bash_command.args)

### [미션 2] 도구가 포함된 에이전트 생성하기

제공된 범용 도구 리스트 `tools_chatbot`을 에이전트 선언부와 결합시킵니다.

In [ ]:
llm = get_llm(model_name="google_vertexai:gemini-3.5-flash", temperature=0.0)
from app.prompts import CHATBOT_SYSTEM_PROMPT
from app.utils.context import AgentContext
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
agent = create_agent(
    model=llm,
    tools=tools_chatbot,
    system_prompt=CHATBOT_SYSTEM_PROMPT,
    checkpointer=memory,
    context_schema=AgentContext
)

if agent is not None:
    print("✅ 도구 결합형 에이전트 생성 성공!")
else:
    print("❌ 에이전트 생성 실패!")

### [미션 3] 자율 실행 루프(ReAct) 검증

에이전트가 문제를 해결하기 위해 스스로 도구를 식별하고 연계 호출하는지 invoke 함수로 검증해봅니다.

In [ ]:
if agent:
    # 1. 파일 검색 및 읽기 복합 태스크 테스트
    res1 = agent.invoke(
        {"messages": [HumanMessage(content="app 폴더 아래 server.py 파일의 1~15줄 내용을 읽어서 보여줘.")]},
        config={"configurable": {"thread_id": "tools_test_session_001"}}
    )
    print("답변1:", res1["messages"][-1].content)

    res1["messages"][-1].pretty_print()
    
    print("\n" + "="*50 + "\n")
    
    # 2. 시스템 명령어 실행 테스트
    res2 = agent.invoke(
        {"messages": [HumanMessage(content="현재 리눅스 터미널의 파이썬 버전을 bash_command로 확인해서 요약해줘.")]},
        config={"configurable": {"thread_id": "tools_test_session_001"}}
    )
    print("답변2:", res2["messages"][-1].content)
else:
    print("에이전트가 준비되지 않았습니다.")